In [ ]:
"""
Inference-Time Reasoning Agent for CSE476 Final Project
Implements: Chain-of-Thought, Self-Consistency, Self-Verification, and Domain-Specific Prompting
"""

import os
import json
import time
import re
from typing import Dict, List, Any, Optional
from collections import Counter
import requests


"""
Inference-Time Reasoning Agent with Concurrent Processing
"""
import os
import json
import time
import re
from typing import Dict, List, Any, Optional
from collections import Counter
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading


class ReasoningAgent:
    """Agent that uses multiple inference-time techniques to solve reasoning problems."""
    
    def __init__(self, api_key: str = "cse476", 
                 api_base: str = "http://10.4.58.53:41701/v1",
                 model: str = "bens_model"):
        self.api_key = api_key
        self.api_base = api_base
        self.model = model
        self.call_count = 0
        self._lock = threading.Lock()  # Thread safety for call_count
        
    def call_llm(self, prompt: str, system: str = None, 
                 temperature: float = 0.0, max_tokens: int = 2048) -> Optional[str]:
        """Call the LLM API and return the response text."""
        with self._lock:
            self.call_count += 1
        
        url = f"{self.api_base}"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json",
        }
        
        messages = []
        if system:
            messages.append({"role": "system", "content": system})
        messages.append({"role": "user", "content": prompt})
        
        payload = {
            "model": self.model,
            "messages": messages,
            "temperature": temperature,
            "max_tokens": max_tokens,
        }
        
        try:
            resp = requests.post(url, headers=headers, json=payload, timeout=120)
            if resp.status_code == 200:
                data = resp.json()
                return data.get("choices", [{}])[0].get("message", {}).get("content", "")
            else:
                print(f"API Error: {resp.status_code}")
                return None
        except Exception as e:
            print(f"Exception calling LLM: {e}")
            return None
    
    # ========================
    # TECHNIQUE 1: Chain-of-Thought Reasoning
    # ========================
    
    def chain_of_thought(self, question: str, domain: str) -> str:
        """Generate answer using step-by-step reasoning."""
        system = "You are a careful problem solver. Think step-by-step and show your reasoning."
        
        prompt = f"""{question}

Please solve this step-by-step:
1. Understand the problem
2. Plan your approach
3. Work through the solution
4. State the final answer clearly

End with: "Therefore, the answer is: [your answer]"
"""
        
        response = self.call_llm(prompt, system=system, max_tokens=2048)
        if not response:
            return ""
        
        # Extract final answer
        answer = self._extract_answer(response, domain)
        return answer
    
    # ========================
    # TECHNIQUE 2: Self-Consistency (Multiple Samples + Voting)
    # ========================
    
    def self_consistency(self, question: str, domain: str, n_samples: int = 3) -> str:
        """Generate multiple solutions and use majority voting."""
        answers = []
        
        system = "You are a problem solver. Provide a clear final answer."
        
        for i in range(n_samples):
            prompt = f"""{question}

Solve this problem and provide your final answer clearly.
For the final answer, use the format: "Final answer: [your answer]"
"""
            # Use different temperatures for diversity
            temp = 0.3 if i > 0 else 0.0
            response = self.call_llm(prompt, system=system, temperature=temp, max_tokens=2048)
            
            if response:
                answer = self._extract_answer(response, domain)
                if answer:
                    answers.append(answer)
        
        # Majority voting
        if not answers:
            return ""
        
        # For coding/planning, return the longest/most complete answer
        if domain in ['coding', 'planning']:
            return max(answers, key=len)
        
        # For others, use most common answer
        counter = Counter(answers)
        return counter.most_common(1)[0][0]
    
    # ========================
    # TECHNIQUE 3: Self-Verification
    # ========================
    
    def self_verify(self, question: str, initial_answer: str, domain: str) -> str:
        """Verify and potentially correct the initial answer."""
        system = "You are a critical reviewer. Check if the answer is correct."
        
        prompt = f"""Question: {question}

Proposed Answer: {initial_answer}

Verify this answer:
1. Is the reasoning correct?
2. Are there any errors?
3. What is the correct answer?

Provide the verified final answer in the format: "Verified answer: [answer]"
"""
        
        response = self.call_llm(prompt, system=system, max_tokens=2048)
        if not response:
            return initial_answer
        
        verified = self._extract_answer(response, domain)
        return verified if verified else initial_answer
    
    # ========================
    # TECHNIQUE 4: Domain-Specific Prompting
    # ========================
    
    def solve_math(self, question: str) -> str:
        """Specialized solver for math problems using CoT + verification."""
        # Reset local call count for this question
        local_calls = 0
        
        # First attempt with CoT
        answer1 = self.chain_of_thought(question, 'math')
        local_calls += 1
        
        # If we have budget, verify
        if local_calls < 3:
            answer2 = self.self_verify(question, answer1, 'math')
            return answer2
        
        return answer1
    
    def solve_coding(self, question: str) -> str:
        """Specialized solver for coding problems."""
        system = "You are an expert Python programmer. Provide clean, working code."
        
        prompt = f"""{question}

Provide only the code implementation. Do not include the function signature or any explanation.
Just the function body code that should be indented and ready to insert.
"""
        
        response = self.call_llm(prompt, system=system, max_tokens=2048)
        if not response:
            return ""
        
        # Clean up the code
        code = response.strip()
        # Remove markdown code blocks if present
        code = re.sub(r'^```python\n', '', code)
        code = re.sub(r'^```\n', '', code)
        code = re.sub(r'\n```$', '', code)
        
        return code
    
    def solve_common_sense(self, question: str) -> str:
        """Specialized solver for common sense questions."""
        system = "You are a knowledgeable assistant. Provide concise, accurate answers."
        
        prompt = f"""{question}

Provide a brief, direct answer. Just the key information needed.
Answer:"""
        
        response = self.call_llm(prompt, system=system, max_tokens=2048)
        if not response:
            return ""
        
        # Extract clean answer
        answer = response.strip()
        # Remove "Answer:" prefix if present
        answer = re.sub(r'^Answer:\s*', '', answer, flags=re.IGNORECASE)
        return answer
    
    def solve_planning(self, question: str) -> str:
        """Specialized solver for planning problems."""
        system = "You are a planning expert. Provide a sequence of actions."
        
        prompt = f"""{question}

Provide the action sequence, one action per line, in the format specified in the problem.
"""
        
        response = self.call_llm(prompt, system=system, max_tokens=1024)
        return response.strip() if response else ""
    
    def solve_future_prediction(self, question: str) -> str:
        """Specialized solver for future prediction."""
        system = "You are a forecasting expert. Provide predictions in the exact format requested."
        
        prompt = f"""{question}

Analyze the question carefully and provide your prediction in the EXACT format specified.
Pay attention to whether the answer should be a list, number, or text.
"""
        
        response = self.call_llm(prompt, system=system, max_tokens=2048)
        if not response:
            return ""
        
        answer = self._extract_answer(response, 'future_prediction')
        return answer
    
    # ========================
    # Main Solver with Adaptive Strategy
    # ========================
    
    def solve(self, question: str, domain: str) -> str:
        """Main solving method that dispatches to domain-specific solvers."""
        # Note: call_count is now thread-safe with lock
        
        if domain == 'math':
            answer = self.solve_math(question)
        elif domain == 'coding':
            answer = self.solve_coding(question)
        elif domain == 'common_sense':
            answer = self.self_consistency(question, domain, n_samples=3)
        elif domain == 'planning':
            answer = self.solve_planning(question)
        elif domain == 'future_prediction':
            answer = self.solve_future_prediction(question)
        else:
            # Fallback: use CoT
            answer = self.chain_of_thought(question, domain)
        
        return answer
    
    # ========================
    # Helper Methods
    # ========================
    
    def _extract_answer(self, response: str, domain: str) -> str:
        """Extract the final answer from the model's response."""
        if not response:
            return ""
        
        # For coding, return the whole response
        if domain == 'coding':
            return response.strip()
        
        # For planning, return the whole response
        if domain == 'planning':
            return response.strip()
        
        # For future_prediction, look for list or number format
        if domain == 'future_prediction':
            # Try to find list format
            list_match = re.search(r'\[([^\]]+)\]', response)
            if list_match:
                return list_match.group(0)
            # Otherwise return cleaned response
            return response.strip()
        
        # For math and common_sense, extract concise answer
        patterns = [
            r"(?:Therefore|Thus|Hence|So),?\s+(?:the\s+)?answer\s+is:?\s*(.+?)(?:\n|$)",
            r"Final answer:?\s*(.+?)(?:\n|$)",
            r"Verified answer:?\s*(.+?)(?:\n|$)",
            r"Answer:?\s*(.+?)(?:\n|$)",
        ]
        
        for pattern in patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                answer = match.group(1).strip()
                # Clean up common artifacts
                answer = re.sub(r'[\*\#]+', '', answer)
                answer = answer.strip('.')
                return answer
        
        # Fallback: return last line or first short line
        lines = [l.strip() for l in response.split('\n') if l.strip()]
        if lines:
            # Prefer short lines (likely the answer)
            short_lines = [l for l in lines if len(l) < 100]
            if short_lines:
                return short_lines[-1]
            return lines[-1]
        
        return response.strip()


def main():
    """Main function to process the development data."""
    import json
    
    # Load development data
    with open('cse476_final_project_dev_data.json', 'r') as f:
        data = json.load(f)
    
    # Initialize agent
    agent = ReasoningAgent()
    
    # Get one example from each domain
    examples = {}
    for item in data:
        domain = item['domain']
        if domain not in examples:
            examples[domain] = item
        if len(examples) == 5:
            break
    
    # Test each domain
    results = []
    for domain, item in sorted(examples.items()):
        print(f"\n{'='*80}")
        print(f"Domain: {domain}")
        print(f"Question: {item['input'][:200]}...")
        print(f"Expected: {item['output'][:100]}...")
        
        answer = agent.solve(item['input'], domain)
        
        print(f"Got: {answer[:100]}...")
        print(f"API calls: {agent.call_count}")
        
        results.append({
            'domain': domain,
            'expected': item['output'],
            'predicted': answer,
            'calls': agent.call_count
        })
        
        time.sleep(0.5)
    
    print("\n" + "="*80)
    print(f"Tested {len(results)} examples (1 per domain)")
    
    return results


if __name__ == "__main__":
    results = main()


Domain: coding
Question: Retrieves the names of the repositories of a specified GitHub user, sorted in ascending order by their creation date. The function queries the GitHub API for all repositories of a given user, parses t...
Expected:     response = requests.get(API_URL + user + '/repos')
    data = json.loads(response.text)
    repo...
API Error: 404
Got: ...
API calls: 1

Domain: common_sense
Question: Which magazine was started first Arthur's Magazine or First for Women?...
Expected: Arthur's Magazine...
API Error: 404
API Error: 404
API Error: 404
Got: ...
API calls: 4

Domain: future_prediction
Question: You are an agent that can predict future events. The event to be predicted: "请预测北京时间2025-08-04, 新榜·视频号指数·搞笑·日榜的前3名分别是哪几个视频号？（只回答视频号名称）"
        IMPORTANT: Your final answer MUST end with this exact fo...
Expected: ['辉叔在线', '铁汁妹妹s', '七颗猩猩']...
API Error: 404
Got: ...
API calls: 5

Domain: math
Question: Let $ABCD$ be a convex quadrilateral with $AB = CD = 10$ , $BC = 14$ , and

In [ ]:
def process_single_item(agent, item, index):
    """Process a single item and return result."""
    try:
        answer = agent.solve(item['input'], item['domain'])
        
        return {
            'index': index,
            'domain': item['domain'],
            'input': item['input'],
            'expected': item['output'],
            'predicted': answer,
            'success': True,
            'error': None
        }
    except Exception as e:
        return {
            'index': index,
            'domain': item['domain'],
            'input': item['input'],
            'expected': item['output'],
            'predicted': "",
            'success': False,
            'error': str(e)
        }


def process_batch_concurrent(agent, data, max_workers=50, limit=None):
    """
    Process data concurrently using ThreadPoolExecutor.
    
    Args:
        agent: ReasoningAgent instance
        data: List of data items to process
        max_workers: Number of concurrent threads (default: 5)
        limit: Optional limit on number of items to process
    
    Returns:
        List of results in original order
    """
    if limit:
        data = data[:limit]
    
    results = [None] * len(data)  # Pre-allocate to maintain order
    
    print(f"Processing {len(data)} items with {max_workers} workers...")
    
    # Create thread pool and submit all tasks
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_index = {
            executor.submit(process_single_item, agent, item, idx): idx
            for idx, item in enumerate(data)
        }
        
        # Process completed tasks
        completed = 0
        for future in as_completed(future_to_index):
            result = future.result()
            results[result['index']] = result
            
            completed += 1
            if completed % 10 == 0:
                print(f"Progress: {completed}/{len(data)} ({100*completed/len(data):.1f}%)")
    
    print(f"\nCompleted! Processed {len(results)} items")
    print(f"Total API calls: {agent.call_count}")
    print(f"Average calls per question: {agent.call_count/len(results):.2f}")
    
    return results


def evaluate_results(results):
    """Evaluate results with simple matching."""
    correct = 0
    domain_stats = {}
    errors = 0
    
    for r in results:
        domain = r['domain']
        if domain not in domain_stats:
            domain_stats[domain] = {'total': 0, 'correct': 0}
        
        domain_stats[domain]['total'] += 1
        
        if not r['success']:
            errors += 1
            continue
        
        # Simple normalized comparison
        expected = str(r['expected']).strip().lower()
        predicted = str(r['predicted']).strip().lower()
        
        # For coding/planning, check if predicted is non-empty and reasonable length
        if domain in ['coding', 'planning']:
            if predicted and len(predicted) > 10:
                domain_stats[domain]['correct'] += 1
                correct += 1
        else:
            # Exact or close match
            if expected == predicted or expected in predicted or predicted in expected:
                domain_stats[domain]['correct'] += 1
                correct += 1
    
    # Print statistics
    print("\n" + "="*80)
    print("EVALUATION RESULTS")
    print("="*80)
    print(f"\nOverall Accuracy: {correct}/{len(results)} = {100*correct/len(results):.1f}%")
    print(f"Errors: {errors}")
    print("\nPer-domain breakdown:")
    for domain, stats in sorted(domain_stats.items()):
        acc = 100 * stats['correct'] / stats['total'] if stats['total'] > 0 else 0
        print(f"  {domain:20s}: {stats['correct']:3d}/{stats['total']:3d} = {acc:5.1f}%")
    
    return domain_stats

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm 
def main():
    """Main function with concurrent processing."""
    import json
    
    # Load development data
    with open('cse476_final_project_dev_data.json', 'r') as f:
        data = json.load(f)
    
    # Initialize agent (thread-safe)
    agent = ReasoningAgent()
    
    print("="*80)
    print("CSE476 Final Project - Concurrent Processing")
    print("="*80)
    
    # Process with concurrent requests
    # Start small for testing, increase max_workers as needed
    results = process_batch_concurrent(
        agent, 
        data, 
        max_workers=50,    # Adjust this: 5-10 is reasonable
        limit=50          # Start with 50 for testing
    )
    
    # Evaluate
    stats = evaluate_results(results)
    
    # Save results
    output_file = 'predictions_concurrent.json'
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nResults saved to {output_file}")
    
    return results, stats


if __name__ == "__main__":
    results, stats = main()

CSE476 Final Project - Concurrent Processing
Processing 50 items with 50 workers...
Progress: 50/50 (100.0%)

Completed! Processed 50 items
Total API calls: 50
Average calls per question: 1.00

EVALUATION RESULTS

Overall Accuracy: 10/50 = 20.0%
Errors: 0

Per-domain breakdown:
  math                :  10/ 50 =  20.0%
    Example failures:
      Q: Let $ABCD$ be a convex quadrilateral with $AB = CD = 10$ , $BC = 14$ , and $AD =...
      Expected: 112, Got: 65
      Q: A tennis player computes her win ratio by dividing the number of matches she has...
      Expected: 164, Got: 169

Results saved to predictions_concurrent.json


In [6]:
import json
import requests
import re
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

class ReasoningAgent:
    def __init__(self, api_key, api_base, model):
        self.api_key = api_key
        self.api_base = api_base
        self.model = model
        self.total_calls = 0
        self._lock = threading.Lock()
    
    def call_llm(self, prompt, max_tokens=4096, system_message=None):
        """Call the LLM API"""
        with self._lock:
            self.total_calls += 1
        
        # Truncate prompt if too long (model limit is 8192 tokens)
        # Rough estimate: 4 chars per token, leave room for response
        max_prompt_chars = 20000  # ~5000 tokens, leaving 3000+ for response
        if len(prompt) > max_prompt_chars:
            prompt = prompt[:max_prompt_chars] + "\n\n[Input truncated due to length]"
        
        url = f"{self.api_base}/chat/completions"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        
        messages = []
        if system_message:
            messages.append({"role": "system", "content": system_message})
        messages.append({"role": "user", "content": prompt})
        
        data = {
            "model": self.model,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0.0
        }
        
        try:
            response = requests.post(url, headers=headers, json=data, timeout=120)
            response.raise_for_status()
            result = response.json()
            return result['choices'][0]['message']['content']
        except Exception as e:
            raise Exception(f"API call failed: {str(e)}")
    
    def chain_of_thought(self, question, max_tokens=4096):
        """Chain-of-thought reasoning"""
        prompt = f"""Solve this problem step by step.

Problem: {question}

Think through this carefully, showing your reasoning. At the end, clearly state your final answer.

Therefore, the final answer is:"""
        
        return self.call_llm(prompt, max_tokens=max_tokens)
    
    def self_consistency(self, question, n_samples=3, max_tokens=3072):
        """Generate multiple solutions and take majority vote"""
        responses = []
        for i in range(n_samples):
            prompt = f"""Answer this question. Be direct and concise.

Question: {question}

Answer:"""
            response = self.call_llm(prompt, max_tokens=max_tokens)
            responses.append(response)
        
        # Extract answers from each response
        answers = [self._extract_answer(r, question, 'common_sense') for r in responses]
        
        # Return majority vote
        if answers:
            answer_counts = Counter(answers)
            return answer_counts.most_common(1)[0][0]
        return responses[0]
    
    def _extract_answer_math(self, response, question):
        """Extract numerical answer from math response"""
        # Problem text patterns to avoid
        bad_patterns = [
            r'find\s+', r'\\frac', r'where\s+m\s+and\s+n',
            r'reduced\s+fraction', r'compute', r'calculate',
            r'determine', r'what\s+is', r'how\s+many', r'given\s+that'
        ]
        
        def is_problem_text(line):
            line_lower = line.lower()
            return any(re.search(pattern, line_lower) for pattern in bad_patterns)
        
        # Try explicit answer patterns first
        patterns = [
            r'Therefore,?\s+the\s+final\s+answer\s+is:?\s*(\d+)',
            r'Final\s+answer:?\s*(\d+)',
            r'\\boxed\{(\d+)\}',
            r'answer\s+is:?\s*(\d+)',
            r'=\s*(\d+)\s*$'
        ]
        
        for pattern in patterns:
            matches = re.findall(pattern, response, re.IGNORECASE | re.MULTILINE)
            if matches:
                return matches[-1]
        
        # Look for standalone numbers (not in calculations)
        lines = response.split('\n')
        for line in reversed(lines):
            if is_problem_text(line):
                continue
            
            # Match standalone numbers (1-6 digits, not part of calculation)
            match = re.search(r'(?<![0-9+\-*/=])\b(\d{1,6})\b(?![0-9+\-*/=])', line)
            if match:
                return match.group(1)
        
        # Last resort: any number not in problem text
        all_numbers = re.findall(r'\b(\d{1,6})\b', response)
        for num in reversed(all_numbers):
            # Check if this number appears in problem-like context
            context_pattern = f'.{{0,50}}{re.escape(num)}.{{0,50}}'
            contexts = re.findall(context_pattern, response, re.DOTALL)
            if contexts and not any(is_problem_text(ctx) for ctx in contexts):
                return num
        
        return all_numbers[-1] if all_numbers else ""
    
    def _extract_answer_common_sense(self, response, question):
        """Extract text answer from common sense question - handles bool, str, and text"""
        # Handle boolean responses (True/False questions)
        if isinstance(response, bool):
            return response
        
        # For string responses, check for True/False
        if isinstance(response, str):
            response_lower = response.lower().strip()
            
            # Check for explicit True/False
            if response_lower in ['true', 'yes']:
                return True
            elif response_lower in ['false', 'no']:
                return False
            
            # Look for True/False in response
            if re.search(r'\b(true)\b', response_lower):
                return True
            elif re.search(r'\b(false)\b', response_lower):
                return False
            
            # Look for explicit answer markers
            patterns = [
                r'[Aa]nswer:?\s*(.+?)(?:\n|$)',
                r'[Tt]he answer is:?\s*(.+?)(?:\n|$)',
                r'[Ff]inal answer:?\s*(.+?)(?:\n|$)'
            ]
            
            for pattern in patterns:
                match = re.search(pattern, response)
                if match:
                    answer = match.group(1).strip()
                    # Check if it's True/False
                    answer_lower = answer.lower()
                    if answer_lower in ['true', 'yes']:
                        return True
                    elif answer_lower in ['false', 'no']:
                        return False
                    # Clean up common artifacts
                    answer = re.sub(r'[.!?]$', '', answer)
                    answer = re.sub(r'^\**|\**$', '', answer)
                    return answer.strip()
            
            # Get last non-empty line that's not too long
            lines = [l.strip() for l in response.split('\n') if l.strip()]
            if lines:
                # Prefer shorter lines (likely answers, not explanations)
                candidates = [l for l in lines[-3:] if len(l) < 200]
                if candidates:
                    answer = candidates[-1]
                    answer_lower = answer.lower()
                    if answer_lower in ['true', 'yes']:
                        return True
                    elif answer_lower in ['false', 'no']:
                        return False
                    answer = re.sub(r'[.!?]$', '', answer)
                    return answer.strip()
                return lines[-1]
            
            return response.strip()
        
        return str(response)
    
    def _extract_answer_future_prediction(self, response, question):
        """Extract answer from future prediction - handles list format and boxed format"""
        # Look for list format: ['item1', 'item2'] or [123.45] or ['single']
        list_pattern = r'\[([^\]]+)\]'
        match = re.search(list_pattern, response)
        if match:
            list_content = match.group(0)
            return list_content
        
        # Look for boxed format: \boxed{...}
        boxed_pattern = r'\\boxed\{([^}]+)\}'
        match = re.search(boxed_pattern, response)
        if match:
            answer = match.group(1)
            # Format as list string
            return f"['{answer}']"
        
        # Look for explicit answer markers
        patterns = [
            r'[Aa]nswer:?\s*(.+?)(?:\n|$)',
            r'[Ff]inal answer:?\s*(.+?)(?:\n|$)',
            r'[Pp]rediction:?\s*(.+?)(?:\n|$)'
        ]
        
        for pattern in patterns:
            match = re.search(pattern, response)
            if match:
                answer = match.group(1).strip()
                # If not already in list format, make it one
                if not answer.startswith('['):
                    return f"['{answer}']"
                return answer
        
        # Get last non-empty line
        lines = [l.strip() for l in response.split('\n') if l.strip()]
        if lines:
            answer = lines[-1]
            if not answer.startswith('['):
                return f"['{answer}']"
            return answer
        
        return "['']"
    
    def _extract_answer_coding(self, response, question):
        """Extract ONLY function body (indented lines) from code response"""
        # Remove markdown code fences first
        response = re.sub(r'```(?:python)?\s*\n?', '', response)
        response = re.sub(r'```\s*$', '', response)
        
        # Look for code blocks
        lines = response.split('\n')
        body_lines = []
        in_function = False
        
        for line in lines:
            # Skip import statements and empty lines at start
            if line.strip().startswith('import ') or line.strip().startswith('from '):
                continue
            
            # Skip function definition line
            if line.strip().startswith('def '):
                in_function = True
                continue
            
            # Collect indented lines (function body)
            if in_function and (line.startswith('    ') or line.startswith('\t')):
                body_lines.append(line)
            elif in_function and line.strip():
                # Non-indented line after function started means function ended
                break
        
        if body_lines:
            return '\n'.join(body_lines)
        
        # Fallback: return all indented lines
        indented_lines = [l for l in lines if l.startswith(('    ', '\t')) and l.strip()]
        if indented_lines:
            return '\n'.join(indented_lines)
        
        return response.strip()
    
    def _extract_answer_planning(self, response, question):
        """Extract LISP-style action sequence: (action param1 param2)"""
        lines = response.split('\n')
        action_lines = []
        
        for line in lines:
            line = line.strip()
            
            # Skip preamble/explanation lines
            if any(phrase in line.lower() for phrase in [
                'here is', 'here are', 'the action sequence', 'following',
                'based on', 'to achieve', 'solution:', 'step'
            ]):
                continue
            
            # Skip numbered prefixes
            line = re.sub(r'^\d+\.?\s*', '', line)
            
            # Look for LISP-style format: (action params)
            if line.startswith('(') and ')' in line:
                # Extract just the action in parentheses
                match = re.search(r'\([^)]+\)', line)
                if match:
                    action_lines.append(match.group(0))
        
        if action_lines:
            return '\n'.join(action_lines)
        
        # If no LISP format found, return empty (this means extraction failed)
        return ""
    
    def _extract_answer(self, response, question, domain):
        """Extract answer based on domain"""
        if domain == 'math':
            return self._extract_answer_math(response, question)
        elif domain == 'common_sense':
            return self._extract_answer_common_sense(response, question)
        elif domain == 'future_prediction':
            return self._extract_answer_future_prediction(response, question)
        elif domain == 'coding':
            return self._extract_answer_coding(response, question)
        elif domain == 'planning':
            return self._extract_answer_planning(response, question)
        else:
            return response.strip()
    
    def solve_math(self, question):
        """Solve math problem using chain-of-thought"""
        response = self.chain_of_thought(question)
        return self._extract_answer(response, question, 'math')
    
    def solve_common_sense(self, question):
        """Solve common sense problem - direct answer for True/False, self-consistency for text"""
        # Check if it's a True/False question
        question_lower = question.lower()
        if question_lower.startswith('is ') or question_lower.startswith('are ') or \
           question_lower.startswith('can ') or question_lower.startswith('does ') or \
           question_lower.startswith('do '):
            # Likely True/False question - use direct answer
            prompt = f"""{question}

Answer with True or False only.

Answer:"""
            response = self.call_llm(prompt, max_tokens=512)
            return self._extract_answer(response, question, 'common_sense')
        else:
            # Text answer - use self-consistency
            response = self.self_consistency(question, n_samples=3)
            return self._extract_answer(response, question, 'common_sense')
    
    def solve_coding(self, question):
        """Generate code solution - IMPORTANT: return only function body"""
        prompt = f"""Write a Python function to solve this problem. Return ONLY the function body (the indented lines inside the function), NOT the function definition line or imports.

{question}

Function body (indented lines only):"""
        response = self.call_llm(prompt, max_tokens=2048)
        return self._extract_answer(response, question, 'coding')
    
    def solve_planning(self, question):
        """Generate planning action sequence in LISP format"""
        prompt = f"""{question}

IMPORTANT: Provide the action sequence in LISP format ONLY. Each action must be on a new line in the format:
(action-name parameter1 parameter2 ...)

Do not include explanations, numbering, or natural language. Output only the actions in parentheses.

Actions:"""
        response = self.call_llm(prompt, max_tokens=2048)
        return self._extract_answer(response, question, 'planning')
    
    def solve_future_prediction(self, question):
        """Solve future prediction question - must return list format"""
        prompt = f"""{question}

Answer:"""
        response = self.call_llm(prompt, max_tokens=1024)
        return self._extract_answer(response, question, 'future_prediction')
    
    def solve(self, question, domain):
        """Route to domain-specific solver"""
        if domain == 'math':
            return self.solve_math(question)
        elif domain == 'common_sense':
            return self.solve_common_sense(question)
        elif domain == 'coding':
            return self.solve_coding(question)
        elif domain == 'planning':
            return self.solve_planning(question)
        elif domain == 'future_prediction':
            return self.solve_future_prediction(question)
        else:
            return self.call_llm(question)


def process_single_item(agent, item):
    """Process a single item"""
    try:
        prediction = agent.solve(item['input'], item['domain'])
        
        # Calculate success with normalization
        success = normalize_and_compare(prediction, item['output'], item['domain'])
        
        return {
            'index': item['index'],
            'domain': item['domain'],
            'input': item['input'],
            'expected': item['output'],
            'predicted': prediction,
            'success': success,
            'error': None
        }
    except Exception as e:
        return {
            'index': item['index'],
            'domain': item['domain'],
            'input': item['input'],
            'expected': item['output'],
            'predicted': '',
            'success': False,
            'error': str(e)
        }


def normalize_and_compare(predicted, expected, domain):
    """Compare predicted and expected answers with domain-specific normalization"""
    # Handle boolean types (common_sense True/False)
    if isinstance(expected, bool):
        if isinstance(predicted, bool):
            return predicted == expected
        # Try to convert predicted to bool
        if isinstance(predicted, str):
            pred_lower = str(predicted).lower().strip()
            if pred_lower in ['true', 'yes', '1']:
                return expected == True
            elif pred_lower in ['false', 'no', '0']:
                return expected == False
        return False
    
    # Convert both to strings for comparison
    pred_str = str(predicted).strip()
    exp_str = str(expected).strip()
    
    # Normalize both strings
    pred_norm = pred_str.lower()
    exp_norm = exp_str.lower()
    
    # Exact match
    if pred_norm == exp_norm:
        return True
    
    # For future_prediction: compare list representations
    if domain == 'future_prediction':
        # Both should be string representations of lists
        # Remove whitespace and compare
        pred_compact = re.sub(r'\s+', '', pred_norm)
        exp_compact = re.sub(r'\s+', '', exp_norm)
        if pred_compact == exp_compact:
            return True
    
    # For coding: more lenient comparison (whitespace-normalized)
    if domain == 'coding':
        # Remove all whitespace and compare
        pred_compact = re.sub(r'\s+', '', pred_norm)
        exp_compact = re.sub(r'\s+', '', exp_norm)
        if pred_compact == exp_compact:
            return True
        
        # Very lenient: Check if they're functionally similar (50%+ overlap)
        if len(pred_compact) > 30 and len(exp_compact) > 30:
            # Calculate similarity using longest common substring
            min_len = min(len(pred_compact), len(exp_compact))
            max_len = max(len(pred_compact), len(exp_compact))
            
            # Check if one is substring of other
            if pred_compact in exp_compact or exp_compact in pred_compact:
                overlap = min_len
                similarity = overlap / max_len
                if similarity > 0.5:
                    return True
            
            # Check for significant overlap in key tokens
            # Extract identifiers (variable names, function names)
            pred_tokens = set(re.findall(r'[a-z_][a-z0-9_]*', pred_compact))
            exp_tokens = set(re.findall(r'[a-z_][a-z0-9_]*', exp_compact))
            
            if pred_tokens and exp_tokens:
                overlap = len(pred_tokens & exp_tokens)
                union = len(pred_tokens | exp_tokens)
                jaccard = overlap / union if union > 0 else 0
                
                # If 50%+ of tokens match, consider it correct
                if jaccard >= 0.5:
                    return True
    
    # For planning: normalize action sequences with more lenient comparison
    if domain == 'planning':
        # Remove extra whitespace and normalize
        pred_actions = [l.strip() for l in str(predicted).split('\n') if l.strip()]
        exp_actions = [l.strip() for l in str(expected).split('\n') if l.strip()]
        
        # Normalize each action: lowercase, remove extra spaces
        pred_actions_norm = [re.sub(r'\s+', ' ', a.lower()) for a in pred_actions]
        exp_actions_norm = [re.sub(r'\s+', ' ', a.lower()) for a in exp_actions]
        
        # Exact match
        if pred_actions_norm == exp_actions_norm:
            return True
        
        # More lenient: check if action names match (ignoring parameters)
        # Extract action name from (action-name param1 param2...)
        def extract_action_name(action):
            match = re.match(r'\(([^\s)]+)', action)
            return match.group(1) if match else action
        
        pred_action_names = [extract_action_name(a) for a in pred_actions_norm]
        exp_action_names = [extract_action_name(a) for a in exp_actions_norm]
        
        # If action sequence matches (at least 70% of actions correct)
        if len(pred_action_names) == len(exp_action_names):
            matches = sum(1 for p, e in zip(pred_action_names, exp_action_names) if p == e)
            if matches / len(exp_action_names) >= 0.7:
                return True
    
    # For numbers: compare as integers if possible
    if domain in ['math']:
        try:
            pred_num = int(re.sub(r'[,\s]', '', str(predicted)))
            exp_num = int(re.sub(r'[,\s]', '', str(expected)))
            if pred_num == exp_num:
                return True
        except:
            pass
    
    # Remove punctuation and compare
    pred_clean = re.sub(r'[^\w\s]', '', pred_norm)
    exp_clean = re.sub(r'[^\w\s]', '', exp_norm)
    
    if pred_clean == exp_clean:
        return True
    
    # Check if predicted contains expected or vice versa (for text answers)
    if domain in ['common_sense']:
        if exp_clean in pred_clean or pred_clean in exp_clean:
            # Make sure it's a substantial match (not just one word)
            if len(exp_clean) > 10:
                return True
    
    return False


def process_batch_concurrent(agent, data, max_workers=5):
    """Process batch of examples using concurrent execution"""
    results = [None] * len(data)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_index = {
            executor.submit(process_single_item, agent, item): item['index']
            for item in data
        }
        
        # Process completed tasks
        completed = 0
        for future in as_completed(future_to_index):
            result = future.result()
            results[result['index']] = result
            completed += 1
            
            if completed % 50 == 0:
                print(f"Progress: {completed}/{len(data)} completed")
    
    return results


def evaluate_results(results):
    """Evaluate and print statistics"""
    total = len(results)
    correct = sum(1 for r in results if r['success'])
    errors = sum(1 for r in results if r['error'] is not None)
    
    print(f"\n{'='*60}")
    print(f"OVERALL RESULTS")
    print(f"{'='*60}")
    print(f"Total: {total}")
    print(f"Correct: {correct}")
    print(f"Incorrect: {total - correct - errors}")
    print(f"Errors: {errors}")
    print(f"Accuracy: {correct/total*100:.2f}%")
    
    # Domain breakdown
    from collections import defaultdict
    domain_stats = defaultdict(lambda: {'total': 0, 'correct': 0, 'errors': 0, 'failures': []})
    
    for r in results:
        domain = r['domain']
        domain_stats[domain]['total'] += 1
        if r['success']:
            domain_stats[domain]['correct'] += 1
        elif r['error'] is None:
            domain_stats[domain]['failures'].append((r['index'], r['expected'], r['predicted']))
        if r['error'] is not None:
            domain_stats[domain]['errors'] += 1
    
    print(f"\n{'='*60}")
    print(f"DOMAIN BREAKDOWN")
    print(f"{'='*60}")
    for domain in sorted(domain_stats.keys()):
        stats = domain_stats[domain]
        acc = (stats['correct'] / stats['total'] * 100) if stats['total'] > 0 else 0
        print(f"{domain:20s}: {stats['correct']:3d}/{stats['total']:3d} = {acc:5.2f}% (errors: {stats['errors']})")
        
        # Show a few failures
        if stats['failures'][:3]:
            print(f"  Example failures:")
            for idx, exp, pred in stats['failures'][:3]:
                exp_short = str(exp)[:50] + '...' if len(str(exp)) > 50 else str(exp)
                pred_short = str(pred)[:50] + '...' if len(str(pred)) > 50 else str(pred)
                print(f"    [{idx}] Expected: '{exp_short}' Got: '{pred_short}'")


def main():
    # Configuration
    API_KEY = "cse476"
    API_BASE = "http://10.4.58.53:41701/v1"
    MODEL = "bens_model"
    
    # Initialize agent
    print("Initializing agent...")
    agent = ReasoningAgent(API_KEY, API_BASE, MODEL)
    
    # Load development data
    print("Loading data...")
    with open('./cse476_final_project_dev_data.json', 'r') as f:
        dev_data = json.load(f)
    
    # Add index to each item
    for i, item in enumerate(dev_data):
        item['index'] = i
    
    print(f"Loaded {len(dev_data)} examples")
    print(f"Processing with {5} concurrent workers...")
    
    # Process all examples
    results = process_batch_concurrent(agent, dev_data, max_workers=50)
    
    # Save results
    output_file = 'predictions_all_1000_FINAL.json'
    with open(output_file, 'w') as f:
        json.dump(results, f, indent=2)
    
    print(f"\nResults saved to {output_file}")
    print(f"Total API calls made: {agent.total_calls}")
    print(f"Average calls per question: {agent.total_calls/len(dev_data):.2f}")
    
    # Evaluate
    evaluate_results(results)


if __name__ == "__main__":
    main()

Initializing agent...
Loading data...
Loaded 1000 examples
Processing with 5 concurrent workers...
Progress: 50/1000 completed
Progress: 100/1000 completed
Progress: 150/1000 completed
Progress: 200/1000 completed
Progress: 250/1000 completed
Progress: 300/1000 completed
Progress: 350/1000 completed
Progress: 400/1000 completed
Progress: 450/1000 completed
Progress: 500/1000 completed
Progress: 550/1000 completed
Progress: 600/1000 completed
Progress: 650/1000 completed
Progress: 700/1000 completed
Progress: 750/1000 completed
Progress: 800/1000 completed
Progress: 850/1000 completed
Progress: 900/1000 completed
Progress: 950/1000 completed
Progress: 1000/1000 completed

Results saved to predictions_all_1000_FINAL.json
Total API calls made: 1662
Average calls per question: 1.66

OVERALL RESULTS
Total: 1000
Correct: 230
Incorrect: 754
Errors: 16
Accuracy: 23.00%

DOMAIN BREAKDOWN
coding              :  23/100 = 23.00% (errors: 0)
  Example failures:
    [100] Expected: '    response = r